# ETF / 股票資料更新 + 每日訊號評分｜Colab 排程友善版

這份 Notebook 是執行器。主程式放在 `colab_etf_auto_pipeline.py`，可以手動 Run all，也可以放到 Colab Enterprise Notebook Scheduling。

## 1. 掛載 Google Drive

請先把 `colab_etf_auto_pipeline.py` 與 `requirements_colab_etf.txt` 放到同一個 Drive 資料夾。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 請依你的 Drive 路徑調整
PROJECT_DIR = "/content/drive/MyDrive/ETF_Stock_Auto"


## 2. 安裝套件

In [ ]:
import os, pathlib, subprocess, sys

project_dir = pathlib.Path(PROJECT_DIR)
assert project_dir.exists(), f"找不到 PROJECT_DIR: {PROJECT_DIR}"

req_file = project_dir / "requirements_colab_etf.txt"
assert req_file.exists(), f"找不到 requirements_colab_etf.txt，請放到: {PROJECT_DIR}"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_file)])


## 3. 設定資料庫連線

建議使用 Colab 左側 Secrets 儲存 `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`, `APP_TIMEZONE`。

In [ ]:
import os

try:
    from google.colab import userdata
    for key in ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD", "APP_TIMEZONE"]:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except Exception as e:
    print("Colab Secrets 讀取略過：", e)

# 若沒有使用 Colab Secrets，也可暫時在這裡手動設定，但不建議把帳密留在 Notebook。
# os.environ["DB_HOST"] = "your_host"
# os.environ["DB_PORT"] = "5432"
# os.environ["DB_NAME"] = "your_db"
# os.environ["DB_USER"] = "your_user"
# os.environ["DB_PASSWORD"] = "your_password"

os.environ.setdefault("APP_TIMEZONE", "Asia/Taipei")

missing = [k for k in ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"] if not os.environ.get(k)]
if missing:
    raise RuntimeError("缺少環境變數：" + ", ".join(missing))
else:
    print("✅ DB 環境變數已設定完成。")


## 4. 設定執行參數

正式排程建議使用 `full-today`。補歷史資料時改用 `full-range` 並填日期。

In [ ]:
# 執行模式：
# full-today   = 今日 ETF持股 + 股價/法人/技術指標 + 訊號評分
# full-range   = 區間 ETF持股 + 股價/法人/技術指標 + 訊號評分
# etf-today    = 僅更新今日 ETF 持股
# data-range   = 僅更新區間股價/法人/技術指標
# signal-range = 僅計算區間訊號評分

MODE = "full-today"

# 若 MODE 是 *-today，START_DATE / END_DATE 可留空。
START_DATE = ""
END_DATE = ""

RULE_VERSION = "v1.0"
INCLUDE_EZMONEY = True
INCLUDE_CAPITAL = True
SKIP_SIGNAL = False
CONTINUE_ON_ETF_FAIL = False


## 5. 執行主程式

In [ ]:
import subprocess, sys, pathlib, os

script = pathlib.Path(PROJECT_DIR) / "colab_etf_auto_pipeline.py"
assert script.exists(), f"找不到主程式：{script}"

cmd = [sys.executable, str(script), "--mode", MODE, "--rule-version", RULE_VERSION]

if START_DATE:
    cmd += ["--start-date", START_DATE]
if END_DATE:
    cmd += ["--end-date", END_DATE]
if not INCLUDE_EZMONEY:
    cmd += ["--no-include-ezmoney"]
if not INCLUDE_CAPITAL:
    cmd += ["--no-include-capital"]
if SKIP_SIGNAL:
    cmd += ["--skip-signal"]
if CONTINUE_ON_ETF_FAIL:
    cmd += ["--continue-on-etf-fail"]

print("執行指令：")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f"程式執行失敗，return code = {result.returncode}")


## 6. 排程建議

一般 Colab 適合手動執行與測試；正式排程請用 Colab Enterprise Notebook Scheduling，或把同一支 `colab_etf_auto_pipeline.py` 部署到 Cloud Run Job / VM / GitHub Actions，再由 n8n 觸發。